In [ ]:
# Exportamos las variables de entorno
%env AWS_ACCESS_KEY_ID=minio   
%env AWS_SECRET_ACCESS_KEY=minio123 
%env MLFLOW_S3_ENDPOINT_URL=http://localhost:9000
%env AWS_ENDPOINT_URL_S3=http://localhost:9000

### BÚSQUEDA DE HIPERPARÁMETROS

El dataset ya paso por el proceso de ETL y se encuentra cargado en el S3 bucket, proceso realizado a través de Airflow. Se descarga el set de train y validation para hacer búsqueda de hiperparámetros.

In [ ]:
import mlflow
import awswrangler as wr

# Establecemos la URI de tracking de MLflow
mlflow_server = "http://localhost:5001"
mlflow.set_tracking_uri(mlflow_server)


In [ ]:
import boto3

# --- S3 Configuration ---
s3_endpoint_url = os.getenv("MLFLOW_S3_ENDPOINT_URL") # Use existing env var
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")
bucket_name = 'data' # Make sure this matches your bucket name

# Initialize S3 client
s3_client = boto3.client(
    's3',
    endpoint_url=s3_endpoint_url,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)

def load_csv_from_bucket(bucket_name, key):
    # --- Load x_train.csv from S3 ---
    
    print(f"Attempting to load data from: s3://{bucket_name}/{key}")
    try:
        obj = s3_client.get_object(Bucket=bucket_name, Key=key)
        # Use index_col=0 if your CSV saved the DataFrame index as the first column
        csv = pd.read_csv(io.BytesIO(obj['Body'].read()), index_col=0)
        print("loaded successfully from S3.")
        return csv
    except Exception as e:
        print(f"Error loading x_train.csv from S3: {e}")
        raise
    
    # for para todos los archivos en processed/{key}
        data_dict = {channel: datos}
    
# genererar carpeta train dentro S3, y ahi un archivo por canal. Idem para validation, y test
train_channels = load_csv_from_bucket(bucket_name, 'processed/X_train.csv')
y_train = load_csv_from_bucket(bucket_name, 'processed/y_train.csv').reset_index()

X_val = load_csv_from_bucket(bucket_name, 'processed/X_val.csv')
y_val = load_csv_from_bucket(bucket_name, 'processed/y_val.csv').reset_index()

In [ ]:
import datetime

In [ ]:
# Definir el experimento en MLflow (se crea si no existe)
experiment_name = "HPsearch_experiment"
if not mlflow.get_experiment_by_name(experiment_name):
    mlflow.create_experiment(name=experiment_name)
    
experiment_id = mlflow.get_experiment_by_name(experiment_name)
print(experiment_id)

run_name_parent = "best_hyperparam_"  + datetime.datetime.today().strftime('%Y/%m/%d-%H:%M:%S"')

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch
from mlflow.models.signature import infer_signature
import optuna
from types import SimpleNamespace

# ========================================
# CONFIGURACIÓN GLOBAL
# ========================================
cfg = SimpleNamespace()
cfg.epochs = 200
cfg.seed = 42
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ========================================
# DATASET (tu clase original)
# ========================================
class DiameterDataset(Dataset):
    def __init__(
        self,
        data_dict,
        ch_list,
        window_size,
        min_context,
        max_context,
        feature_scaler=None,
        target_scaler=None,
        fit_scaler=False,
        target="center",
        data_stride=1,
        seed=42
    ):
        if not (window_size <= min_context <= max_context):
            raise ValueError("ERROR: window_size <= min_context <= max_context")
        for val, name in [(window_size, "window_size"), (min_context, "min_context"), (max_context, "max_context")]:
            if val % 2 == 0 or val < 3:
                raise ValueError(f"{name} debe ser impar y >= 3")
        if target != "center":
            raise ValueError("Solo target='center' soportado")

        self.data_dict = data_dict
        self.ch_list = ch_list
        self.window_size = window_size
        self.min_context = min_context
        self.max_context = max_context
        self.data_stride = data_stride
        self.seed = seed
        self.feature_scaler = feature_scaler
        self.target_scaler = target_scaler

        self.feature_cols = ['Axial', 'Temperature', 'Flux']
        self.target_col = 'MeanRateOutDiam'

        self.feats_padded_list = []
        self.targets_list = []
        self.sample_indices = []
        self.lengths = []

        max_half = max_context // 2

        if fit_scaler:
            all_feats = []
            all_targs = []
            for ch in ch_list:
                df = data_dict[ch]
                feats = df[self.feature_cols].values.astype(np.float32)
                targs = df[self.target_col].values.astype(np.float32)
                all_feats.append(feats)
                all_targs.append(targs)
            all_feats = np.vstack(all_feats)
            all_targs = np.hstack(all_targs).reshape(-1, 1)
            self.feature_scaler = MinMaxScaler().fit(all_feats)
            self.target_scaler = MinMaxScaler().fit(all_targs)

        for ch_idx, ch in enumerate(ch_list):
            df = data_dict[ch]
            feats = df[self.feature_cols].values.astype(np.float32)
            targs = df[self.target_col].values.astype(np.float32)

            if self.feature_scaler:
                feats = self.feature_scaler.transform(feats)
            if self.target_scaler:
                targs = self.target_scaler.transform(targs.reshape(-1, 1)).flatten()

            feats_padded_np = np.pad(feats, ((max_half, max_half), (0, 0)), mode='edge')
            feats_padded = torch.from_numpy(feats_padded_np)
            targs_tensor = torch.from_numpy(targs)

            self.feats_padded_list.append(feats_padded)
            self.targets_list.append(targs_tensor)
            self.lengths.append(len(df))

            L = len(df)
            for i in range(0, L, data_stride):
                self.sample_indices.append((ch_idx, i))

    def __len__(self):
        return len(self.sample_indices)

    def __getitem__(self, idx):
        ch_idx, center_idx = self.sample_indices[idx]
        feats_padded = self.feats_padded_list[ch_idx]
        targets = self.targets_list[ch_idx]

        worker_info = torch.utils.data.get_worker_info()
        if worker_info is not None:
            base_seed = worker_info.id + self.seed + idx
        else:
            base_seed = self.seed + idx
        rng = random.Random(base_seed)

        context_size = rng.randint(self.min_context, self.max_context)
        if context_size % 2 == 0:
            context_size -= 1
        half = context_size // 2

        offset = self.max_context // 2
        center_padded = center_idx + offset
        start = center_padded - half
        end = center_padded + half + 1

        window_real = feats_padded[start:end]

        if context_size > self.window_size:
            step = context_size / self.window_size
            ds_idx = torch.round(torch.arange(0, context_size, step)).long()[:self.window_size]
            window_final = window_real[ds_idx]
        else:
            window_final = window_real

        axial_vals = window_final[:, 0]
        axial_center = axial_vals[self.window_size // 2]
        delta_left = axial_center - axial_vals[0] if self.window_size > 1 else 0
        delta_right = axial_vals[-1] - axial_center if self.window_size > 1 else 0

        sample_info = (
            self.ch_list[ch_idx],
            center_idx,
            context_size,
            (axial_vals[0].item(), axial_vals[-1].item()),
            float(delta_left),
            float(delta_right)
        )

        return window_final, targets[center_idx], sample_info

def custom_collate_fn(batch):
    features = torch.stack([item[0] for item in batch])
    targets = torch.stack([item[1] for item in batch])
    sample_info = [item[2] for item in batch]
    return features, targets, sample_info

# ========================================
# MODELO CNN1D
# ========================================
def conv_block(in_channels, out_channels, kernel_size=3, dropout=0.0):
    padding = kernel_size // 2
    return nn.Sequential(
        nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding),
        nn.BatchNorm1d(out_channels),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout) if dropout > 0 else nn.Identity()
    )

class CNN1DRegressor(nn.Module):
    def __init__(self, input_channels=3, nlayers=3, dropout=0.2):
        super().__init__()
        layers = []
        out_ch = 32
        in_ch = input_channels
        for i in range(nlayers):
            layers.append(conv_block(in_ch, out_ch, kernel_size=3, dropout=dropout))
            in_ch = out_ch
            if i < nlayers - 1:
                out_ch *= 2
        self.conv_blocks = nn.Sequential(*layers)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(in_ch, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)              # (B, W, 4) → (B, 4, W)
        x = self.conv_blocks(x)
        x = self.global_pool(x).flatten(1)
        x = self.fc(x)
        return x.squeeze(-1)

# ========================================
# CARGA DE DATOS (AJUSTA TU RUTA O S3)
# ========================================
# Ejemplo con S3 (descomenta y ajusta)
# import boto3
# s3 = boto3.client('s3')
# obj = s3.get_object(Bucket='tu-bucket', Key='data.csv')
# df_full = pd.read_csv(obj['Body'])
# data_by_ch = {ch: group.drop(columns=['channel']) for ch, group in df_full.groupby('channel')}

# data_by_ch = ...  # diccionario {channel_name: pd.DataFrame}

# Listas de canales
train_channels_list = [...]   # lista real
val_channels_list = [...]
test_channels_list = [...]

train_channels_aug = train_channels_list * 3

# Dataset inicial solo para obtener scalers
train_dataset_base = DiameterDataset(
    data_dict=data_by_ch,
    ch_list=train_channels_aug,
    window_size=21,
    min_context=21,
    max_context=121,
    fit_scaler=True,
    target="center",
    data_stride=1,
    seed=42
)

# ========================================
# FUNCIÓN OBJETIVO OPTUNA
# ========================================
def objective(trial):
    # Hiperparámetros
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    nlayers = trial.suggest_int("nlayers", 2, 6)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
    window_size = trial.suggest_int("window_size", 11, 41, step=2)  # impar
    max_context = trial.suggest_int("max_context", window_size + 20, 201, step=20)
    scheduler_factor = trial.suggest_float("scheduler_factor", 0.1, 0.9)
    scheduler_patience = trial.suggest_int("scheduler_patience", 3, 10)

    # Datasets del trial
    train_dataset_trial = DiameterDataset(
        data_dict=data_by_ch,
        ch_list=train_channels_aug,
        window_size=window_size,
        min_context=window_size,
        max_context=max_context,
        feature_scaler=train_dataset_base.feature_scaler,
        target_scaler=train_dataset_base.target_scaler,
        target="center",
        data_stride=1,
        seed=42
    )
    val_dataset_trial = DiameterDataset(
        data_dict=data_by_ch,
        ch_list=val_channels_list,
        window_size=window_size,
        min_context=window_size,
        max_context=max_context,
        feature_scaler=train_dataset_base.feature_scaler,
        target_scaler=train_dataset_base.target_scaler,
        target="center",
        data_stride=1
    )

    train_loader = DataLoader(train_dataset_trial, batch_size=batch_size, shuffle=True, drop_last=True,
                              num_workers=0, pin_memory=True, collate_fn=custom_collate_fn)
    val_loader = DataLoader(val_dataset_trial, batch_size=batch_size, shuffle=False, drop_last=True,
                            collate_fn=custom_collate_fn)

    model = CNN1DRegressor(input_channels=3, nlayers=nlayers, dropout=dropout).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                          factor=scheduler_factor, patience=scheduler_patience)

    # Entrenamiento con early stopping
    patience_es = 20
    best_val_loss = float('inf')
    counter = 0
    best_state = None

    for epoch in range(200):
        model.train()
        for X, y, _ in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

        # Validación
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X, y, _ in val_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                val_loss += criterion(out, y).item() * X.size(0)
        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            counter = 0
        else:
            counter += 1
            if counter >= patience_es:
                break

    model.load_state_dict(best_state)
    return best_val_loss

# ========================================
# BÚSQUEDA OPTUNA + MLFLOW
# ========================================
mlflow.set_experiment("Diameter_CNN1D_HyperSearch")

with mlflow.start_run(run_name="Optuna_Parent_Run"):
    mlflow.log_param("n_trials", 60)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=20)
    )
    study.optimize(objective, n_trials=60)

    best_params = study.best_params
    best_val_loss = study.best_value

    mlflow.log_metric("best_val_loss", best_val_loss)
    mlflow.log_params({f"best_{k}": v for k, v in best_params.items()})

    print("=== MEJORES PARÁMETROS ===")
    for k, v in best_params.items():
        print(f"{k}: {v}")

    # ========================================
    # ENTRENAMIENTO FINAL CON MEJORES PARÁMETROS
    # ========================================
    with mlflow.start_run(run_name="Final_Model_BestParams", nested=True):
        # Datasets finales
        final_train_ds = DiameterDataset(
            data_dict=data_by_ch,
            ch_list=train_channels_aug,
            window_size=best_params["window_size"],
            min_context=best_params["window_size"],
            max_context=best_params["max_context"],
            feature_scaler=train_dataset_base.feature_scaler,
            target_scaler=train_dataset_base.target_scaler,
            target="center",
            data_stride=1,
            seed=42
        )
        final_val_ds = DiameterDataset(
            data_dict=data_by_ch,
            ch_list=val_channels_list,
            window_size=best_params["window_size"],
            min_context=best_params["window_size"],
            max_context=best_params["max_context"],
            feature_scaler=train_dataset_base.feature_scaler,
            target_scaler=train_dataset_base.target_scaler,
            target="center",
            data_stride=1
        )
        final_test_ds = DiameterDataset(
            data_dict=data_by_ch,
            ch_list=test_channels_list,
            window_size=best_params["window_size"],
            min_context=best_params["window_size"],
            max_context=best_params["max_context"],
            feature_scaler=train_dataset_base.feature_scaler,
            target_scaler=train_dataset_base.target_scaler,
            target="center",
            data_stride=1
        )

        train_loader = DataLoader(final_train_ds, batch_size=best_params["batch_size"], shuffle=True,
                                  drop_last=True, num_workers=0, pin_memory=True, collate_fn=custom_collate_fn)
        val_loader = DataLoader(final_val_ds, batch_size=best_params["batch_size"], shuffle=False,
                                drop_last=True, collate_fn=custom_collate_fn)
        test_loader = DataLoader(final_test_ds, batch_size=1024, shuffle=False, drop_last=False)

        model = CNN1DRegressor(input_channels=3, nlayers=best_params["nlayers"],
                               dropout=best_params["dropout"]).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"],
                                     weight_decay=best_params["weight_decay"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=best_params["scheduler_factor"],
            patience=best_params["scheduler_patience"])
        criterion = nn.MSELoss()

        # Entrenamiento final
        train_losses, val_losses = [], []
        for epoch in range(cfg.epochs):
            model.train()
            train_loss = 0.0
            for X, y, _ in train_loader:
                X, y = X.to(device), y.to(device)
                optimizer.zero_grad()
                out = model(X)
                loss = criterion(out, y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * X.size(0)
            train_loss /= len(train_loader.dataset)
            train_losses.append(train_loss)

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for X, y, _ in val_loader:
                    X, y = X.to(device), y.to(device)
                    out = model(X)
                    val_loss += criterion(out, y).item() * X.size(0)
            val_loss /= len(val_loader.dataset)
            val_losses.append(val_loss)
            scheduler.step(val_loss)

            print(f"Epoch {epoch+1}/{cfg.epochs} - Train: {train_loss:.6f} - Val: {val_loss:.6f}")

        # Log métricas entrenamiento
        mlflow.log_metric("final_train_loss", train_losses[-1])
        mlflow.log_metric("final_val_loss", val_losses[-1])

        # Curva de pérdida
        fig, ax = plt.subplots()
        ax.plot(train_losses, label="Train")
        ax.plot(val_losses, label="Val")
        ax.legend()
        ax.set_xlabel("Epoch")
        ax.set_ylabel("MSE Loss")
        mlflow.log_figure(fig, "loss_curve.png")
        plt.close(fig)

        # Evaluación en test
        model.eval()
        preds_scaled, targets_scaled = [], []
        preds_orig, targets_orig = [], []

        with torch.no_grad():
            for X, y, _ in test_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                preds_scaled.extend(out.cpu().numpy())
                targets_scaled.extend(y.cpu().numpy())

                if train_dataset_base.target_scaler:
                    p_orig = train_dataset_base.target_scaler.inverse_transform(
                        np.array(out.cpu()).reshape(-1, 1)).flatten()
                    t_orig = train_dataset_base.target_scaler.inverse_transform(
                        np.array(y.cpu()).reshape(-1, 1)).flatten()
                    preds_orig.extend(p_orig)
                    targets_orig.extend(t_orig)

        # Métricas
        mse_scaled = mean_squared_error(targets_scaled, preds_scaled)
        rmse_scaled = np.sqrt(mse_scaled)
        mae_scaled = mean_absolute_error(targets_scaled, preds_scaled)
        r2_scaled = r2_score(targets_scaled, preds_scaled)

        mse = mean_squared_error(targets_orig, preds_orig)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(targets_orig, preds_orig)
        r2 = r2_score(targets_orig, preds_orig)

        metrics = {
            "test_mse_scaled": mse_scaled, "test_rmse_scaled": rmse_scaled,
            "test_mae_scaled": mae_scaled, "test_r2_scaled": r2_scaled,
            "test_mse": mse, "test_rmse": rmse, "test_mae": mae, "test_r2": r2
        }
        mlflow.log_metrics(metrics)

        # Log modelo
        sample_input, _, _ = next(iter(train_loader))
        signature = infer_signature(sample_input.numpy(), model(sample_input.to(device)).cpu().numpy())

        mlflow.pytorch.log_model(
            pytorch_model=model,
            artifact_path="model",
            signature=signature,
            registered_model_name="diameter_cnn1d_prod",
            metadata={"model_version": 1}
        )

        model_uri = mlflow.get_artifact_uri("model")
        print(f"Modelo final guardado en: {model_uri}")

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()

# Nombre del modelo registrado (ajusta si quieres otro)
registered_model_name = "diameter_cnn1d_prod"
description = "CNN1D Regressor para predecir MeanRateOutDiam usando contexto variable y downsampling"

# Crear el modelo registrado si no existe
try:
    client.create_registered_model(name=registered_model_name, description=description)
    print(f"Modelo registrado creado: {registered_model_name}")
except mlflow.exceptions.RestException:
    # Ya existe, actualizar descripción si quieres
    client.update_registered_model(name=registered_model_name, description=description)
    print(f"Modelo registrado ya existía: {registered_model_name}")

# Tags relevantes para la versión
tags = {
    "model_type": "CNN1DRegressor",
    "framework": "PyTorch",
    "task": "regression",
    "target": "MeanRateOutDiam",
    "window_size": best_params["window_size"],
    "max_context": best_params["max_context"],
    "nlayers": best_params["nlayers"],
    "dropout": best_params["dropout"],
    "lr": best_params["lr"],
    "batch_size": best_params["batch_size"],
    "weight_decay": best_params["weight_decay"],
    "scheduler_factor": best_params["scheduler_factor"],
    "scheduler_patience": best_params["scheduler_patience"],
    "train_augmentation": 3,
    "data_stride": 1,
    "seed": cfg.seed,
    "test_rmse": rmse,           # métrica en escala original
    "test_mae": mae,
    "test_r2": r2,
    "test_rmse_scaled": rmse_scaled,
    "final_val_loss": val_losses[-1],
}

# Obtener el run_id y source del run actual
current_run = mlflow.active_run()
run_id = current_run.info.run_id
source = f"runs:/{run_id}/model"  # Ruta estándar para artefactos en el run actual

# Crear nueva versión del modelo
model_version = client.create_model_version(
    name=registered_model_name,
    source=source,
    run_id=run_id,
    tags=tags
)

print(f"Nueva versión creada: {model_version.version}")

# Asignar alias "champion" a esta versión (sobrescribe si ya existe)
client.set_registered_model_alias(
    name=registered_model_name,
    alias="champion",
    version=model_version.version
)